# Data and Linguistic Analysis

Dataset preparation, user splits, linguistic features, statistical tests, and sensitivity analyses. ASD labels follow the dataset categories.


## 1. Configuration

Experiment paths, seeds, and execution options.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from IPython.display import display
BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
MULTISEED = BASE / '17_conference_revision/multiseed'
STACKING = BASE / '17_conference_revision/stacking_oof'
SEEDS = [42, 43, 44, 45, 46]
ENABLE_TRAINING = False
ENABLE_META_FIT = False
ENABLE_THRESHOLD_SEARCH = False
TRAINING_STAGE = None
METRICS = ['accuracy', 'precision_ASD', 'recall_ASD', 'f1_ASD', 'macro_f1', 'recall_non_ASD', 'f1_non_ASD', 'roc_auc']

def require_files(paths):
    missing = [str(Path(p)) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError('Missing files:\n' + '\n'.join(missing))

def read_json(path):
    require_files([path])
    return json.loads(Path(path).read_text())

def partition(data, seed):
    folder = MULTISEED / f'seed_{seed}'
    require_files([folder / f'{part}_users.csv' for part in ['train', 'val', 'test']])
    parts = {}
    for part in ['train', 'val', 'test']:
        names = pd.read_csv(folder / f'{part}_users.csv')['username']
        assert names.notna().all() and names.is_unique
        parts[part] = set(names)
    assert tuple((len(parts[p]) for p in ['train', 'val', 'test'])) == (146, 37, 61)
    assert not (parts['train'] & parts['val'] or parts['train'] & parts['test'] or parts['val'] & parts['test'])
    assert set().union(*parts.values()) == set(data['username'])
    positions = {p: np.flatnonzero(data['username'].isin(names).to_numpy()) for p, names in parts.items()}
    labels = data.iloc[positions['test']].groupby('username', sort=True)['label'].first()
    assert labels.value_counts().to_dict() == {1: 54, 0: 7}
    return (parts, positions)


In [2]:
ENABLE_PREPROCESSING = False
ENABLE_ANALYSIS = True
PREPARATION_STAGE = None
ANALYSIS_STAGE = 'tweet_count'

def require_source_stage(stage, mode, outputs=(), inputs=()):
    choices = {
        'preprocessing': (ENABLE_PREPROCESSING, PREPARATION_STAGE),
        'analysis': (ENABLE_ANALYSIS, ANALYSIS_STAGE),
        'training': (ENABLE_TRAINING, TRAINING_STAGE)
    }
    enabled, selected = choices[mode]
    if enabled is not True or selected != stage:
        raise RuntimeError(f'{stage} is disabled. Select its stage and enable {mode} explicitly.')
    existing = []
    for pattern in outputs:
        if any(character in pattern for character in '*?['):
            existing.extend(str(path) for path in BASE.glob(pattern))
        elif (BASE/pattern).exists():
            existing.append(str(BASE/pattern))
    if existing:
        raise FileExistsError('Stage outputs already exist:\n'+'\n'.join(existing))
    require_files([BASE/path for path in inputs])


In [3]:
if not (BASE/'02_preprocessing/cleaned_data.pkl').is_file():
    try:
        from google.colab import drive
    except ImportError:
        raise FileNotFoundError(f'Dataset path: {BASE}') from None
    drive.mount('/content/drive_research')


Mounted at /content/drive_research


## 2. Dataset Preparation

Raw CSVs use semicolon delimiters. Optional order: dataset -> holdout_split -> multiseed_splits. Stages are disabled; existing artifacts are never overwritten.


In [4]:
def prepare_dataset():
    require_source_stage('dataset', 'preprocessing', outputs=['02_preprocessing/df_preprocessed.csv', '02_preprocessing/preprocess_report.csv', '02_preprocessing/cleaned_data.pkl'], inputs=[])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    import os
    import glob
    import pandas as pd
    DATASET_PATH = '/content/drive_research/MyDrive/thesis/dataset'
    SAVE_DIR = '/content/drive_research/MyDrive/thesis/saved_results'
    DATA_DIR = os.path.join(SAVE_DIR, '01_dataset')
    os.makedirs(DATA_DIR, exist_ok=True)
    RAW_SAVE_PATH = os.path.join(DATA_DIR, 'df_raw.csv')
    if os.path.exists(RAW_SAVE_PATH):
        df_raw = pd.read_csv(RAW_SAVE_PATH)
    else:
        all_files = sorted(glob.glob(os.path.join(DATASET_PATH, '*.csv')))
        if len(all_files) == 0:
            raise ValueError('Required data or compatible artifacts are unavailable.')
        dfs = []
        zero_row_files = []
        for f in all_files:
            try:
                temp = pd.read_csv(f, sep=';')
                if len(temp) == 0:
                    zero_row_files.append(os.path.basename(f))
                    continue
                dfs.append(temp)
            except Exception as e:
                pass
        if len(dfs) == 0:
            raise ValueError('Required data or compatible artifacts are unavailable.')
        df_raw = pd.concat(dfs, ignore_index=True)
        df_raw['label'] = (df_raw['Type'] == 'Paciente').astype(int)
        df_raw.to_csv(RAW_SAVE_PATH, index=False)
    PREPROCESS_SAVE_PATH = os.path.join(SAVE_DIR, '02_preprocessing', 'df_preprocessed.csv')
    REPORT_SAVE_PATH = os.path.join(SAVE_DIR, '02_preprocessing', 'preprocess_report.csv')
    os.makedirs(os.path.dirname(PREPROCESS_SAVE_PATH), exist_ok=True)

    def preprocess_tweet(text):
        if not isinstance(text, str):
            return ''
        text = html.unescape(text)
        text = re.sub('https?://\\S+|www\\.\\S+', ' ', text)
        text = re.sub('^RT\\s+', '', text)
        text = re.sub('@\\w+', ' ', text)
        text = text.replace('#', '')
        text = re.sub('[^\\w\\s!?]', ' ', text)
        text = text.lower()
        text = re.sub('\\s+', ' ', text).strip()
        return text
    if os.path.exists(PREPROCESS_SAVE_PATH):
        df = pd.read_csv(PREPROCESS_SAVE_PATH)
        preprocess_report = pd.read_csv(REPORT_SAVE_PATH)
    else:
        df = df_raw.copy()
        stats_rows = []
        stats_rows.append(('Raw data', len(df), df['username'].nunique()))
        df = df[df['lang'].eq('en')].copy()
        stats_rows.append(('English only', len(df), df['username'].nunique()))
        df['cleaned_tweet'] = df['tweet'].apply(preprocess_tweet)
        df = df[df['cleaned_tweet'].str.len() > 0].copy()
        stats_rows.append(('Remove empty cleaned text', len(df), df['username'].nunique()))
        df = df.drop_duplicates(subset=['username', 'cleaned_tweet'], keep='first').reset_index(drop=True)
        stats_rows.append(('Remove user-level duplicate cleaned text', len(df), df['username'].nunique()))
        preprocess_report = pd.DataFrame(stats_rows, columns=['stage', 'tweets', 'users'])
        df.to_csv(PREPROCESS_SAVE_PATH, index=False)
        preprocess_report.to_csv(REPORT_SAVE_PATH, index=False)
    display(preprocess_report)
    CLEANED_DATA_PATH = os.path.join(SAVE_DIR, '02_preprocessing', 'cleaned_data.pkl')
    df.to_pickle(CLEANED_DATA_PATH)
    CLEANED_DATA_PATH = os.path.join(SAVE_DIR, '02_preprocessing', 'cleaned_data.pkl')
    if not 'df' in locals():
        df = pd.read_pickle(CLEANED_DATA_PATH)
    labels_per_user = df.groupby('username')['label'].nunique()
    assert labels_per_user.max() == 1, 'Artifact validation failed.'
    assert df['cleaned_tweet'].str.len().gt(0).all(), 'Artifact validation failed.'
    assert set(df['label'].unique()) == {0, 1}, 'Artifact validation failed.'
    return preprocess_report


In [5]:
def prepare_holdout_split():
    require_source_stage('holdout_split', 'preprocessing', outputs=['03_split/train_users.csv', '03_split/val_users.csv', '03_split/test_users.csv'], inputs=['02_preprocessing/cleaned_data.pkl'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    df = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')
    from sklearn.model_selection import train_test_split
    SPLIT_DIR = os.path.join(SAVE_DIR, '03_split')
    os.makedirs(SPLIT_DIR, exist_ok=True)
    TRAIN_USERS_PATH = os.path.join(SPLIT_DIR, 'train_users.csv')
    VAL_USERS_PATH = os.path.join(SPLIT_DIR, 'val_users.csv')
    TEST_USERS_PATH = os.path.join(SPLIT_DIR, 'test_users.csv')
    if os.path.exists(TRAIN_USERS_PATH) and os.path.exists(VAL_USERS_PATH) and os.path.exists(TEST_USERS_PATH):
        train_users = pd.read_csv(TRAIN_USERS_PATH)
        val_users = pd.read_csv(VAL_USERS_PATH)
        test_users = pd.read_csv(TEST_USERS_PATH)
    else:
        user_table = df.groupby('username', as_index=False)['label'].first()
        train_pool_users, test_users = train_test_split(user_table, test_size=0.25, random_state=SEED, stratify=user_table['label'])
        train_users, val_users = train_test_split(train_pool_users, test_size=0.2, random_state=SEED, stratify=train_pool_users['label'])
        train_users.to_csv(TRAIN_USERS_PATH, index=False)
        val_users.to_csv(VAL_USERS_PATH, index=False)
        test_users.to_csv(TEST_USERS_PATH, index=False)
    train_df_full = df[df['username'].isin(train_users['username'])].copy()
    val_df = df[df['username'].isin(val_users['username'])].copy()
    test_df = df[df['username'].isin(test_users['username'])].copy()
    for name, part in [('Train', train_df_full), ('Val', val_df), ('Test', test_df)]:
        pass
    train_set = set(train_df_full['username'])
    val_set = set(val_df['username'])
    test_set = set(test_df['username'])
    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)
    all_users = set(df['username'])
    split_users = train_set | val_set | test_set
    assert all_users == split_users
    return pd.DataFrame([{'split': n, 'users': f['username'].nunique(), 'tweets': len(f)} for n, f in [('train', train_df_full), ('val', val_df), ('test', test_df)]])


In [6]:
def prepare_multiseed_splits():
    require_source_stage('multiseed_splits', 'preprocessing', outputs=['17_conference_revision/multiseed/seed_*/*_users.csv', '17_conference_revision/multiseed/split_summary.csv'], inputs=['02_preprocessing/cleaned_data.pkl', '03_split/train_users.csv', '03_split/val_users.csv', '03_split/test_users.csv'])
    from pathlib import Path
    import pandas as pd
    from sklearn.model_selection import train_test_split
    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    OUT = BASE / '17_conference_revision/multiseed'
    OUT.mkdir(parents=True, exist_ok=True)
    df = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')[['username', 'label']]
    assert df.groupby('username')['label'].nunique().eq(1).all()
    users = df.groupby('username', as_index=False)['label'].first()
    assert len(users) == 244
    split_plans = {}
    summary = []
    for seed in [42, 43, 44, 45, 46]:
        if seed == 42:
            parts = {}
            for part in ['train', 'val', 'test']:
                names = pd.read_csv(BASE / f'03_split/{part}_users.csv')['username']
                assert set(names).issubset(set(users['username']))
                parts[part] = users[users['username'].isin(names)].copy()
        else:
            pool, test = train_test_split(users, test_size=0.25, stratify=users['label'], random_state=seed)
            train, val = train_test_split(pool, test_size=0.2, stratify=pool['label'], random_state=seed)
            parts = {'train': train, 'val': val, 'test': test}
        tr = set(parts['train']['username'])
        va = set(parts['val']['username'])
        te = set(parts['test']['username'])
        assert not (tr & va or tr & te or va & te)
        assert tr | va | te == set(users['username'])
        folder = OUT / f'seed_{seed}'
        folder.mkdir(exist_ok=True)
        for part, table in parts.items():
            table.to_csv(folder / f'{part}_users.csv', index=False)
        split_plans[seed] = parts
        summary.append({'seed': seed, 'train_users': len(parts['train']), 'val_users': len(parts['val']), 'test_users': len(parts['test']), 'test_non_ASD': int((parts['test']['label'] == 0).sum()), 'test_ASD': int((parts['test']['label'] == 1).sum())})
    summary = pd.DataFrame(summary)
    summary.to_csv(OUT / 'split_summary.csv', index=False)
    display(summary)
    return summary


In [7]:
stage_functions = {
    'dataset': prepare_dataset,
    'holdout_split': prepare_holdout_split,
    'multiseed_splits': prepare_multiseed_splits
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


## Dataset and Saved Splits

Load cleaned_tweet and the recorded user partitions.


In [8]:
require_files([BASE/'02_preprocessing/cleaned_data.pkl'])
data = pd.read_pickle(BASE/'02_preprocessing/cleaned_data.pkl')[
    ['username','label','cleaned_tweet']
].reset_index(drop=True)
assert data.notna().all().all()
assert data['label'].isin([0,1]).all()
assert data.groupby('username')['label'].nunique().eq(1).all()
assert data['username'].nunique() == 244
partitions = {seed:partition(data,seed) for seed in SEEDS}
display(pd.DataFrame([{'tweets':len(data),'users':data['username'].nunique()}]))


,tweets,users
0,920070,244


### Dataset Overview

Class counts, tweet activity, text length, and split sizes. Word counts use whitespace; balanced sizes are counts before model fitting.


In [9]:
REVISION = BASE / '17_conference_revision'
LINGUISTIC = REVISION / 'linguistic_robustness'
TWEET_COUNT_RESULTS = LINGUISTIC / 'tweet_count_sensitivity.json'
TWEET_COUNT_FEATURES = ['tweet_length_sd', 'tweet_length_iqr', 'vocab_richness_sd',
                        'vocab_richness_iqr', 'pronoun_i_rate_iqr',
                        'sentiment_subjectivity_iqr']
TWEET_COUNT_COLUMNS = ['feature', 'coef_label', 'p_label', 'coef_log_n']
REPORT_BASE_MODELS = ['Logistic Regression', 'SVM', 'Random Forest', 'Bi-LSTM',
                      'TextCNN', 'DistilBERT', 'RoBERTa', 'BERTweet']

def read_report_csv(path, **kwargs):
    require_files([path])
    return pd.read_csv(path, **kwargs)

def show_result(title, function):
    print(title)
    try:
        table = function()
    except FileNotFoundError as error:
        print(str(error))
        return None
    display(table)
    return table


def dataset_overview(data):
    users = data.groupby('username', sort=True).agg(label=('label','first'),
                                                   n_tweets=('label','size'))
    tweets = data.groupby('label').size()
    counts = users.groupby('label').size()
    overview = pd.DataFrame({'tweets':tweets,'users':counts}).reindex([1,0])
    overview['tweet_fraction'] = overview['tweets']/len(data)
    overview['user_fraction'] = overview['users']/len(users)
    overview.index = ['ASD','non-ASD']
    display(overview.round(4))
    checks = pd.DataFrame([{
        'tweets':len(data), 'users':len(users),
        'missing_values':int(data.isna().sum().sum()),
        'empty_texts':int(data['cleaned_tweet'].str.strip().eq('').sum()),
        'duplicate_user_text_rows':int(data.duplicated(['username','cleaned_tweet']).sum())
    }])
    display(checks)
    activity = users.groupby('label')['n_tweets'].describe(percentiles=[.25,.5,.75,.95])
    activity.index = activity.index.map({0:'non-ASD',1:'ASD'})
    display(activity.round(2))
    words = data['cleaned_tweet'].str.count(r'\S+')
    lengths = pd.DataFrame({'label':data['label'], 'whitespace_words':words})
    lengths = lengths.groupby('label')['whitespace_words'].describe(percentiles=[.25,.5,.75,.95])
    lengths.index = lengths.index.map({0:'non-ASD',1:'ASD'})
    display(lengths.round(2))
    del words, lengths

def split_overview(data, partitions):
    rows = []
    for seed in SEEDS:
        _, positions = partitions[seed]
        for split in ['train','val','test']:
            frame = data.iloc[positions[split]]
            users = frame.groupby('username')['label'].first()
            tweets = frame['label'].value_counts().reindex([0,1],fill_value=0)
            row = {'seed':seed, 'split':split, 'users':len(users), 'tweets':len(frame),
                   'ASD_users':int(users.eq(1).sum()), 'non_ASD_users':int(users.eq(0).sum()),
                   'ASD_tweets':int(tweets[1]), 'non_ASD_tweets':int(tweets[0])}
            if split=='train':
                row['balanced_tweets_per_class'] = int(tweets.min())
                row['balanced_training_tweets'] = int(2*tweets.min())
            rows.append(row)
    return pd.DataFrame(rows)

def equal_count_eligibility(data):
    settings = read_report_csv(LINGUISTIC/'equal_count_settings.csv').iloc[0]
    users = data.groupby('username').agg(label=('label','first'),n_tweets=('label','size'))
    eligible = users.loc[users['n_tweets'].ge(int(settings['tweets_per_user']))]
    return pd.DataFrame([{'tweets_per_user':int(settings['tweets_per_user']),
                          'eligible_users':len(eligible),
                          'ASD_users':int(eligible['label'].eq(1).sum()),
                          'non_ASD_users':int(eligible['label'].eq(0).sum()),
                          'excluded_users':len(users)-len(eligible)}])

def raw_counts():
    path = BASE / '01_dataset/df_raw.csv'
    user_ids, n_tweets = set(), 0
    if path.is_file():
        for chunk in pd.read_csv(path, usecols=['username'], chunksize=100000):
            n_tweets += len(chunk)
            user_ids.update(chunk['username'].dropna())
    else:
        dataset_dir = BASE.parent / 'dataset'
        files = sorted(dataset_dir.glob('*.csv'))
        if not files:
            raise FileNotFoundError(
                f'Raw counts require {path} or semicolon-delimited CSVs in {dataset_dir}.'
            )
        used, empty, unreadable = 0, 0, 0
        has_username = False
        for file in files:
            try:
                frame = pd.read_csv(file, sep=';')
            except Exception:
                unreadable += 1
                continue
            if len(frame) == 0:
                empty += 1
                continue
            used += 1
            n_tweets += len(frame)
            if 'username' in frame.columns:
                has_username = True
                user_ids.update(frame['username'].dropna())
        if used == 0:
            raise ValueError('No nonempty readable raw CSVs are available.')
        if not has_username:
            raise ValueError('Raw CSVs do not contain a username column.')
        if empty or unreadable:
            print(f'Raw CSVs: {used} used, {empty} empty, {unreadable} unreadable.')
    return pd.DataFrame([{'raw_tweets': n_tweets, 'raw_users': len(user_ids)}])


In [10]:
dataset_overview(data)
show_result('Raw Dataset',raw_counts)
show_result('User-Disjoint Splits',lambda:split_overview(data,partitions))


,tweets,users,tweet_fraction,user_fraction
ASD,838199,215,0.911,0.8811
non-ASD,81871,29,0.089,0.1189


,tweets,users,missing_values,empty_texts,duplicate_user_text_rows
0,920070,244,0,0,0


,count,mean,std,min,25%,50%,75%,95%,max
label,,,,,,,,,
non-ASD,29.0,2823.14,1779.91,237.0,1267.0,2828.0,4698.0,4928.2,5237.0
ASD,215.0,3898.60,1584.15,12.0,3080.0,4686.0,4939.0,5328.6,5403.0


,count,mean,std,min,25%,50%,75%,95%,max
label,,,,,,,,,
non-ASD,81871.0,16.15,13.03,1.0,6.0,12.0,22.0,45.0,68.0
ASD,838199.0,19.07,14.99,1.0,7.0,14.0,28.0,49.0,99.0


Raw Dataset
Raw CSVs: 245 used, 7 empty, 0 unreadable.


,raw_tweets,raw_users
0,1014723,245


User-Disjoint Splits


,seed,split,users,tweets,ASD_users,non_ASD_users,ASD_tweets,non_ASD_tweets,balanced_tweets_per_class,balanced_training_tweets
0,42,train,146,541706,128,18,489122,52584,52584.0,105168.0
1,42,val,37,128924,33,4,118896,10028,NaN,NaN
2,42,test,61,249440,54,7,230181,19259,NaN,NaN
3,43,train,146,544748,128,18,488039,56709,56709.0,113418.0
4,43,val,37,156071,33,4,144456,11615,NaN,NaN
5,43,test,61,219251,54,7,205704,13547,NaN,NaN
6,44,train,146,574609,128,18,519089,55520,55520.0,111040.0
7,44,val,37,116284,33,4,108832,7452,NaN,NaN
8,44,test,61,229177,54,7,210278,18899,NaN,NaN
9,45,train,146,567591,128,18,515135,52456,52456.0,104912.0


,seed,split,users,tweets,ASD_users,non_ASD_users,ASD_tweets,non_ASD_tweets,balanced_tweets_per_class,balanced_training_tweets
0,42,train,146,541706,128,18,489122,52584,52584.0,105168.0
1,42,val,37,128924,33,4,118896,10028,NaN,NaN
2,42,test,61,249440,54,7,230181,19259,NaN,NaN
3,43,train,146,544748,128,18,488039,56709,56709.0,113418.0
4,43,val,37,156071,33,4,144456,11615,NaN,NaN
5,43,test,61,219251,54,7,205704,13547,NaN,NaN
6,44,train,146,574609,128,18,519089,55520,55520.0,111040.0
7,44,val,37,116284,33,4,108832,7452,NaN,NaN
8,44,test,61,229177,54,7,210278,18899,NaN,NaN
9,45,train,146,567591,128,18,515135,52456,52456.0,104912.0


## 3. Linguistic Features and Tests

Nine tweet features; user means, sample SD, and IQR. Welch tests for means; Mann-Whitney tests for dispersion, with separate FDR families.


In [11]:
def extract_linguistic_features():
    require_source_stage('features', 'preprocessing', outputs=['16_linguistic_consistency/tweet_linguistic_features.pkl', '16_linguistic_consistency/tweet_chunks/chunk_*.pkl'], inputs=['02_preprocessing/cleaned_data.pkl'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    import os, re, gc, glob, math
    import numpy as np
    import pandas as pd
    from textblob import TextBlob
    import textstat
    LING_DIR = f'{SAVE_DIR}/16_linguistic_consistency'
    CHUNK_DIR = f'{LING_DIR}/tweet_chunks'
    TWEET_FEAT_PATH = f'{LING_DIR}/tweet_linguistic_features.pkl'
    CLEANED_PATH = f'{SAVE_DIR}/02_preprocessing/cleaned_data.pkl'
    os.makedirs(CHUNK_DIR, exist_ok=True)
    FEATURES = ['tweet_length', 'vocab_richness', 'sentiment_polarity', 'sentiment_subjectivity', 'pronoun_i_rate', 'pronoun_we_rate', 'exclamation_rate', 'question_rate', 'readability']
    CHUNK_SIZE = 10000

    def extract_tweet_features(text):
        text = str(text) if pd.notna(text) else ''
        text = text.strip()
        lower_text = text.lower()
        tokens = re.findall('\\b\\w+\\b', lower_text)
        word_count = len(tokens)
        vocab_richness = len(set(tokens)) / word_count if word_count > 0 else np.nan
        if text:
            try:
                sentiment = TextBlob(text).sentiment
                polarity = float(sentiment.polarity)
                subjectivity = float(sentiment.subjectivity)
            except Exception:
                polarity, subjectivity = (np.nan, np.nan)
            try:
                readability = float(textstat.flesch_reading_ease(text))
            except Exception:
                readability = np.nan
        else:
            polarity, subjectivity, readability = (np.nan, np.nan, np.nan)
        if word_count > 0:
            pronoun_i_rate = len(re.findall('\\bi\\b', lower_text)) / word_count
            pronoun_we_rate = len(re.findall('\\bwe\\b', lower_text)) / word_count
            exclamation_rate = text.count('!') / word_count
            question_rate = text.count('?') / word_count
        else:
            pronoun_i_rate = pronoun_we_rate = np.nan
            exclamation_rate = question_rate = np.nan
        return {'tweet_length': word_count, 'vocab_richness': vocab_richness, 'sentiment_polarity': polarity, 'sentiment_subjectivity': subjectivity, 'pronoun_i_rate': pronoun_i_rate, 'pronoun_we_rate': pronoun_we_rate, 'exclamation_rate': exclamation_rate, 'question_rate': question_rate, 'readability': readability}
    if os.path.exists(TWEET_FEAT_PATH):
        tweet_feat = pd.read_pickle(TWEET_FEAT_PATH)
    else:
        ref = pd.read_pickle(CLEANED_PATH)[['username', 'label', 'cleaned_tweet']]
        ref = ref.reset_index(drop=True)
        n_chunks = math.ceil(len(ref) / CHUNK_SIZE)
        for i in range(n_chunks):
            chunk_path = f'{CHUNK_DIR}/chunk_{i:04d}.pkl'
            if os.path.exists(chunk_path):
                continue
            part = ref.iloc[i * CHUNK_SIZE:(i + 1) * CHUNK_SIZE]
            feats = pd.DataFrame([extract_tweet_features(t) for t in part['cleaned_tweet']])
            out = pd.concat([part[['username', 'label']].reset_index(drop=True), feats], axis=1)
            out.to_pickle(chunk_path)
            del feats, out
            gc.collect()
        tweet_feat = pd.concat([pd.read_pickle(p) for p in sorted(glob.glob(f'{CHUNK_DIR}/chunk_*.pkl'))], ignore_index=True)
        tweet_feat.to_pickle(TWEET_FEAT_PATH)
    return pd.DataFrame([{'tweets': len(tweet_feat), 'users': tweet_feat['username'].nunique()}])


In [12]:
def summarize_user_features():
    require_source_stage('user_features', 'preprocessing', outputs=['16_linguistic_consistency/user_linguistic_consistency.csv'], inputs=['16_linguistic_consistency/tweet_linguistic_features.pkl'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    FEATURES = ['tweet_length', 'vocab_richness', 'sentiment_polarity', 'sentiment_subjectivity', 'pronoun_i_rate', 'pronoun_we_rate', 'exclamation_rate', 'question_rate', 'readability']
    LING_DIR = f'{SAVE_DIR}/16_linguistic_consistency'
    tweet_feat = pd.read_pickle(BASE / '16_linguistic_consistency/tweet_linguistic_features.pkl')
    USER_CONS_PATH = f'{LING_DIR}/user_linguistic_consistency.csv'

    def iqr_func(x):
        return x.quantile(0.75) - x.quantile(0.25)
    grp = tweet_feat.groupby(['username', 'label'])[FEATURES]
    user_mean = grp.mean().add_suffix('_mean')
    user_sd = grp.std(ddof=1).add_suffix('_sd')
    user_iqr = grp.agg(iqr_func).add_suffix('_iqr')
    user_n = grp.size().rename('n_tweets')
    user_cons = pd.concat([user_n, user_mean, user_sd, user_iqr], axis=1).reset_index()
    n_single = (user_cons['n_tweets'] < 2).sum()
    user_cons.to_csv(USER_CONS_PATH, index=False)
    display(user_cons.groupby('label')[['n_tweets']].describe().round(1))
    return user_cons.drop(columns='username').groupby('label').mean()


In [13]:
def compute_mean_statistics():
    require_source_stage('mean_statistics', 'analysis', outputs=['05_evaluation/linguistic_statistics.csv'], inputs=['16_linguistic_consistency/tweet_linguistic_features.pkl'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    FEATURE_SAVE_PATH = os.path.join(SAVE_DIR, '02_preprocessing', 'linguistic_features.pkl')
    if os.path.exists(FEATURE_SAVE_PATH):
        feat_df = pd.read_pickle(FEATURE_SAVE_PATH)
    else:
        if 'tweet_feat' not in locals():
            tweet_feat = pd.read_pickle(f'{SAVE_DIR}/16_linguistic_consistency/tweet_linguistic_features.pkl')
        FEAT_COLS = [c for c in tweet_feat.columns if c not in ['username', 'label']]
        feat_df = tweet_feat.groupby(['username', 'label'])[FEAT_COLS].mean().reset_index()
        feat_df.to_pickle(FEATURE_SAVE_PATH)
    display(feat_df.drop(columns='username').groupby('label').mean().T.rename(columns={0: 'non-ASD', 1: 'ASD'}))
    from scipy import stats
    from statsmodels.stats.multitest import multipletests
    FEATURE_SAVE_PATH = os.path.join(SAVE_DIR, '02_preprocessing', 'linguistic_features.pkl')
    STATS_SAVE_PATH = os.path.join(SAVE_DIR, '05_evaluation', 'linguistic_statistics.csv')
    os.makedirs(os.path.dirname(STATS_SAVE_PATH), exist_ok=True)
    if 'feat_df' not in locals():
        feat_df = pd.read_pickle(FEATURE_SAVE_PATH)

    def hedges_g(x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        nx, ny = (len(x), len(y))
        sx2 = np.var(x, ddof=1)
        sy2 = np.var(y, ddof=1)
        pooled_sd = np.sqrt(((nx - 1) * sx2 + (ny - 1) * sy2) / (nx + ny - 2))
        if pooled_sd == 0:
            return 0.0
        d = (np.mean(x) - np.mean(y)) / pooled_sd
        correction = 1 - 3 / (4 * (nx + ny) - 9)
        return d * correction
    feature_cols = [c for c in feat_df.columns if c not in ['username', 'label']]
    rows = []
    for col in feature_cols:
        asd = feat_df.loc[feat_df['label'] == 1, col].dropna()
        non_asd = feat_df.loc[feat_df['label'] == 0, col].dropna()
        t_stat, p_value = stats.ttest_ind(asd, non_asd, equal_var=False)
        rows.append({'feature': col, 'ASD_mean': asd.mean(), 'non_ASD_mean': non_asd.mean(), 'difference': asd.mean() - non_asd.mean(), 't_stat': t_stat, 'p_value': p_value, 'hedges_g': hedges_g(asd, non_asd)})
    linguistic_stats = pd.DataFrame(rows)
    linguistic_stats['p_FDR'] = multipletests(linguistic_stats['p_value'], method='fdr_bh')[1]
    linguistic_stats['significant_FDR_005'] = linguistic_stats['p_FDR'] < 0.05
    linguistic_stats.to_csv(STATS_SAVE_PATH, index=False)
    display(linguistic_stats.sort_values('p_FDR').round(4))
    return linguistic_stats


In [14]:
def compute_dispersion_statistics():
    require_source_stage('dispersion_statistics', 'analysis', outputs=['16_linguistic_consistency/linguistic_consistency_statistics.csv'], inputs=['16_linguistic_consistency/user_linguistic_consistency.csv'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    FEATURES = ['tweet_length', 'vocab_richness', 'sentiment_polarity', 'sentiment_subjectivity', 'pronoun_i_rate', 'pronoun_we_rate', 'exclamation_rate', 'question_rate', 'readability']
    LING_DIR = f'{SAVE_DIR}/16_linguistic_consistency'
    user_cons = pd.read_csv(BASE / '16_linguistic_consistency/user_linguistic_consistency.csv')
    from scipy.stats import mannwhitneyu
    from statsmodels.stats.multitest import multipletests
    CONS_STATS_PATH = f'{LING_DIR}/linguistic_consistency_statistics.csv'

    def rank_biserial(u_stat, n1, n2):
        return 2 * u_stat / (n1 * n2) - 1
    rows = []
    for feat in FEATURES:
        for measure in ['sd', 'iqr']:
            col = f'{feat}_{measure}'
            asd = user_cons.loc[user_cons['label'] == 1, col].dropna()
            non = user_cons.loc[user_cons['label'] == 0, col].dropna()
            u_stat, p_val = mannwhitneyu(asd, non, alternative='two-sided')
            rows.append({'feature': feat, 'measure': measure.upper(), 'n_ASD': len(asd), 'n_nonASD': len(non), 'median_ASD': asd.median(), 'median_nonASD': non.median(), 'U': u_stat, 'p_value': p_val, 'rank_biserial_r': rank_biserial(u_stat, len(asd), len(non))})
    cons_stats = pd.DataFrame(rows)
    cons_stats['p_FDR'] = multipletests(cons_stats['p_value'], method='fdr_bh')[1]
    cons_stats['significant_FDR_005'] = cons_stats['p_FDR'] < 0.05
    cons_stats.to_csv(CONS_STATS_PATH, index=False)
    display(cons_stats.sort_values('p_FDR').round(4))
    return cons_stats


In [15]:
stage_functions = {
    'features': extract_linguistic_features,
    'user_features': summarize_user_features,
    'mean_statistics': compute_mean_statistics,
    'dispersion_statistics': compute_dispersion_statistics
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


### Linguistic Features - Tables I-II

Group means, within-user dispersion, effect sizes, and FDR-adjusted tests. Load the statistical result tables.


In [16]:
def mean_results():
    table = read_report_csv(BASE / '05_evaluation/linguistic_statistics.csv')
    assert len(table) == 9 and table['feature'].is_unique
    display(pd.DataFrame([{'mean_comparisons': len(table),
                          'FDR_significant': int(table['significant_FDR_005'].sum())}]))
    return table.round(4)
def dispersion_results():
    table = read_report_csv(LINGUISTIC / 'dispersion_all_244_users.csv')
    assert len(table) == 18 and not table.duplicated(['feature', 'measure']).any()
    display(pd.DataFrame([{'dispersion_comparisons': len(table),
                          'FDR_significant': int(table['significant_FDR_005'].sum())}]))
    return table.round(4)


In [17]:
show_result('Mean Linguistic Features - Table I',mean_results)
show_result('Within-User Dispersion - Table II',dispersion_results)


Mean Linguistic Features - Table I


,mean_comparisons,FDR_significant
0,9,2


,feature,ASD_mean,non_ASD_mean,difference,t_stat,p_value,hedges_g,p_FDR,significant_FDR_005
0,tweet_length,18.4894,16.0323,2.4571,2.9983,0.0042,0.4173,0.0190,True
1,vocab_richness,0.9370,0.9474,-0.0104,-3.1784,0.0026,-0.4451,0.0190,True
2,sentiment_polarity,0.1233,0.1524,-0.0292,-1.9601,0.0578,-0.3904,0.1734,False
3,sentiment_subjectivity,0.3938,0.3956,-0.0019,-0.1440,0.8863,-0.0285,0.9017,False
4,pronoun_i_rate,0.0320,0.0268,0.0052,1.6750,0.1035,0.4099,0.2330,False
5,pronoun_we_rate,0.0037,0.0040,-0.0003,-0.8134,0.4208,-0.1363,0.6311,False
6,exclamation_rate,0.0503,0.0490,0.0013,0.1243,0.9017,0.0220,0.9017,False
7,question_rate,0.0143,0.0124,0.0019,0.9809,0.3323,0.1580,0.5981,False
8,readability,69.2016,68.3472,0.8544,0.4988,0.6206,0.0852,0.7980,False


Within-User Dispersion - Table II


,dispersion_comparisons,FDR_significant
0,18,6


,feature,measure,n_ASD,n_nonASD,median_ASD,median_nonASD,U,p_value,rank_biserial_r,p_FDR,significant_FDR_005
0,tweet_length,SD,215,29,14.0416,11.3984,4324.0,0.0007,0.3870,0.0130,True
1,vocab_richness,SD,215,29,0.0785,0.0725,4147.0,0.0039,0.3302,0.0178,True
2,sentiment_polarity,SD,215,29,0.2968,0.3155,2435.0,0.0559,-0.2189,0.1119,False
3,sentiment_subjectivity,SD,215,29,0.3160,0.3269,2396.0,0.0433,-0.2314,0.0974,False
4,pronoun_i_rate,SD,215,29,0.0530,0.0462,3740.0,0.0813,0.1997,0.1463,False
5,pronoun_we_rate,SD,215,29,0.0175,0.0192,2731.0,0.2793,-0.1240,0.4190,False
6,exclamation_rate,SD,215,29,0.1473,0.1422,3361.0,0.4958,0.0781,0.6865,False
7,question_rate,SD,215,29,0.0572,0.0490,3652.0,0.1345,0.1715,0.2201,False
8,readability,SD,215,29,26.4276,26.9144,3013.0,0.7707,-0.0335,0.8160,False
9,tweet_length,IQR,215,29,19.0000,14.0000,4145.5,0.0039,0.3298,0.0178,True


,feature,measure,n_ASD,n_nonASD,median_ASD,median_nonASD,U,p_value,rank_biserial_r,p_FDR,significant_FDR_005
0,tweet_length,SD,215,29,14.0416,11.3984,4324.0,0.0007,0.3870,0.0130,True
1,vocab_richness,SD,215,29,0.0785,0.0725,4147.0,0.0039,0.3302,0.0178,True
2,sentiment_polarity,SD,215,29,0.2968,0.3155,2435.0,0.0559,-0.2189,0.1119,False
3,sentiment_subjectivity,SD,215,29,0.3160,0.3269,2396.0,0.0433,-0.2314,0.0974,False
4,pronoun_i_rate,SD,215,29,0.0530,0.0462,3740.0,0.0813,0.1997,0.1463,False
5,pronoun_we_rate,SD,215,29,0.0175,0.0192,2731.0,0.2793,-0.1240,0.4190,False
6,exclamation_rate,SD,215,29,0.1473,0.1422,3361.0,0.4958,0.0781,0.6865,False
7,question_rate,SD,215,29,0.0572,0.0490,3652.0,0.1345,0.1715,0.2201,False
8,readability,SD,215,29,26.4276,26.9144,3013.0,0.7707,-0.0335,0.8160,False
9,tweet_length,IQR,215,29,19.0000,14.0000,4145.5,0.0039,0.3298,0.0178,True


## 4. Readability

Readability text retains sentence punctuation. Tweet ID, row position, username, and label checks align the corrected feature table.


In [18]:
def compute_revised_readability():
    require_source_stage('readability', 'preprocessing', outputs=['readability_revision/tweet_readability_revised.pkl', 'readability_revision/user_readability_revised.csv'], inputs=['02_preprocessing/cleaned_data.pkl'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    import pandas as pd
    check_df = pd.read_pickle(f'{SAVE_DIR}/02_preprocessing/cleaned_data.pkl')
    if 'tweet' in check_df.columns:
        pass
    import os
    import re
    import html
    import numpy as np
    import pandas as pd
    import textstat
    from tqdm.auto import tqdm
    textstat.set_lang('en')
    df_read = check_df.reset_index(drop=True)

    def prepare_readability_text(value):
        if pd.isna(value):
            return ''
        text = html.unescape(str(value))
        text = re.sub('https?://\\S+|www\\.\\S+', ' ', text)
        text = re.sub('^RT\\s+', '', text)
        text = re.sub('@\\w+', ' ', text)
        text = text.replace('#', '')
        text = re.sub('[^\\w\\s.!?]', ' ', text)
        return re.sub('\\s+', ' ', text).strip()

    def calculate_readability(value):
        text = prepare_readability_text(value)
        if not re.search('\\w', text):
            return np.nan
        return float(textstat.flesch_reading_ease(text))
    readability_new = df_read[['id', 'username', 'label']].copy()
    readability_new['row_position'] = np.arange(len(df_read))
    readability_new['readability'] = [calculate_readability(text) for text in tqdm(df_read['tweet'], total=len(df_read), desc='Analysis')]
    revision_dir = f'{SAVE_DIR}/readability_revision'
    os.makedirs(revision_dir, exist_ok=True)
    readability_new.to_pickle(f'{revision_dir}/tweet_readability_revised.pkl')
    user_readability = readability_new.groupby(['username', 'label'])['readability'].agg(readability_mean='mean', readability_sd='std', readability_iqr=lambda x: x.quantile(0.75) - x.quantile(0.25)).reset_index()
    user_readability.to_csv(f'{revision_dir}/user_readability_revised.csv', index=False)
    return user_readability.drop(columns='username').groupby('label').mean()


In [19]:
def compute_revised_user_statistics():
    require_source_stage('revised_user_statistics', 'analysis', outputs=['readability_revision/user_linguistic_revised.csv', 'readability_revision/mean_statistics_revised.csv', 'readability_revision/variability_statistics_revised.csv'], inputs=['16_linguistic_consistency/user_linguistic_consistency.csv', 'readability_revision/user_readability_revised.csv'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    import os
    import numpy as np
    import pandas as pd
    from scipy.stats import ttest_ind, mannwhitneyu
    from statsmodels.stats.multitest import multipletests
    revision_dir = f'{SAVE_DIR}/readability_revision'
    old = pd.read_csv(f'{SAVE_DIR}/16_linguistic_consistency/user_linguistic_consistency.csv', dtype={'username': str})
    new = pd.read_csv(f'{revision_dir}/user_readability_revised.csv', dtype={'username': str})
    read_cols = ['readability_mean', 'readability_sd', 'readability_iqr']
    revised = old.drop(columns=read_cols).merge(new[['username', 'label'] + read_cols], on=['username', 'label'], how='outer', validate='one_to_one', indicator=True)
    assert revised['_merge'].eq('both').all(), 'Artifact validation failed.'
    assert len(revised) == len(old) == 244
    assert np.isfinite(revised[read_cols].to_numpy()).all(), 'Artifact validation failed.'
    revised = revised.drop(columns='_merge')
    features = ['tweet_length', 'vocab_richness', 'sentiment_polarity', 'sentiment_subjectivity', 'pronoun_i_rate', 'pronoun_we_rate', 'exclamation_rate', 'question_rate', 'readability']
    mean_rows = []
    for feature in features:
        col = f'{feature}_mean'
        a = revised.loc[revised['label'] == 1, col].dropna()
        b = revised.loc[revised['label'] == 0, col].dropna()
        t, p = ttest_ind(a, b, equal_var=False)
        df = len(a) + len(b) - 2
        pooled_sd = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / df)
        correction = 1 - 3 / (4 * df - 1)
        g = correction * (a.mean() - b.mean()) / pooled_sd
        mean_rows.append({'feature': feature, 'ASD_mean': a.mean(), 'non_ASD_mean': b.mean(), 'difference': a.mean() - b.mean(), 't_stat': t, 'p_value': p, 'hedges_g': g})
    mean_stats = pd.DataFrame(mean_rows)
    var_rows = []
    for feature in features:
        for measure in ['sd', 'iqr']:
            col = f'{feature}_{measure}'
            a = revised.loc[revised['label'] == 1, col].dropna()
            b = revised.loc[revised['label'] == 0, col].dropna()
            u, p = mannwhitneyu(a, b, alternative='two-sided')
            var_rows.append({'feature': feature, 'measure': measure.upper(), 'median_ASD': a.median(), 'median_nonASD': b.median(), 'U': u, 'p_value': p, 'rank_biserial_r': 2 * u / (len(a) * len(b)) - 1})
    var_stats = pd.DataFrame(var_rows)
    for table in [mean_stats, var_stats]:
        assert np.isfinite(table['p_value']).all()
        table['p_FDR'] = multipletests(table['p_value'], method='fdr_bh')[1]
        table['significant_FDR_005'] = table['p_FDR'] < 0.05
    revised.to_csv(f'{revision_dir}/user_linguistic_revised.csv', index=False)
    mean_stats.to_csv(f'{revision_dir}/mean_statistics_revised.csv', index=False)
    var_stats.to_csv(f'{revision_dir}/variability_statistics_revised.csv', index=False)
    display(mean_stats.sort_values('p_FDR').round(4))
    display(var_stats.sort_values('p_FDR').round(4))
    return mean_stats


In [20]:
def align_revised_tweet_features():
    require_source_stage('corrected_features', 'preprocessing', outputs=['17_conference_revision/linguistic_robustness/tweet_linguistic_features_corrected.pkl'], inputs=['02_preprocessing/cleaned_data.pkl', '16_linguistic_consistency/tweet_linguistic_features.pkl', 'readability_revision/tweet_readability_revised.pkl'])
    features_check = pd.read_pickle(BASE / '16_linguistic_consistency/tweet_linguistic_features.pkl')
    readability_revised = pd.read_pickle(BASE / 'readability_revision/tweet_readability_revised.pkl')
    import numpy as np
    import pandas as pd
    from pathlib import Path
    base = Path('/content/drive_research/MyDrive/thesis/saved_results')
    source = pd.read_pickle(base / '02_preprocessing/cleaned_data.pkl').reset_index(drop=True)
    old_features = features_check.reset_index(drop=True).copy()
    revised = readability_revised.copy()
    assert len(source) == len(old_features) == len(revised) == 920070
    assert revised['row_position'].notna().all()
    assert revised['row_position'].is_unique
    assert np.array_equal(np.sort(revised['row_position'].to_numpy()), np.arange(len(source))), 'Artifact validation failed.'
    revised = revised.sort_values('row_position').reset_index(drop=True)
    for column in ['username', 'label']:
        assert np.array_equal(old_features[column].to_numpy(), source[column].to_numpy()), 'Artifact validation failed.'
        assert np.array_equal(revised[column].to_numpy(), source[column].to_numpy()), 'Artifact validation failed.'
    assert source['id'].notna().all() and revised['id'].notna().all()
    assert np.array_equal(source['id'].astype(str).to_numpy(), revised['id'].astype(str).to_numpy()), 'Artifact validation failed.'
    assert np.isfinite(revised['readability'].to_numpy()).all(), 'Artifact validation failed.'
    features_corrected = old_features.copy()
    features_corrected['readability'] = revised['readability'].to_numpy()
    pd.testing.assert_frame_equal(features_corrected.drop(columns='readability'), old_features.drop(columns='readability'))
    summary = features_corrected.groupby(['username', 'label'])['readability'].mean().groupby('label').agg(['count', 'mean'])
    assert np.isclose(summary.loc[0, 'mean'], 74.0304, atol=0.0001, rtol=0)
    assert np.isclose(summary.loc[1, 'mean'], 75.1459, atol=0.0001, rtol=0)
    output_dir = base / '17_conference_revision/linguistic_robustness'
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / 'tweet_linguistic_features_corrected.pkl'
    features_corrected.to_pickle(output_path)
    return summary


In [21]:
stage_functions = {
    'readability': compute_revised_readability,
    'revised_user_statistics': compute_revised_user_statistics,
    'corrected_features': align_revised_tweet_features
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


## 5. Sensitivity Analyses

Rank-transformed tweet-count regression; 500 tweets/user over 100 repeats; MATTR windows 25/50/100 over 20 repeats; noun-based NMF10 and topic-adjusted HC3 regression.


In [22]:
def validate_tweet_count_record(record):
    import math
    expected = {
        'schema_version': 1,
        'analysis': 'tweet_count_sensitivity',
        'source_notebook': '00_Data_and_Linguistic_Analysis.ipynb',
        'source_function': 'compute_tweet_count_sensitivity',
        'formula': 'y_rank ~ label + log_n',
        'round_decimals': 4,
    }
    if not isinstance(record, dict) or any(record.get(k) != v for k, v in expected.items()):
        raise ValueError('Invalid tweet-count sensitivity metadata.')
    rows = record.get('rows')
    if not isinstance(rows, list) or len(rows) != len(TWEET_COUNT_FEATURES):
        raise ValueError('Expected six tweet-count sensitivity rows.')
    for feature, row in zip(TWEET_COUNT_FEATURES, rows):
        if not isinstance(row, dict) or set(row) != set(TWEET_COUNT_COLUMNS) or row['feature'] != feature:
            raise ValueError('Invalid tweet-count sensitivity columns or feature order.')
        for column in TWEET_COUNT_COLUMNS[1:]:
            value = row[column]
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError('Tweet-count sensitivity estimates must be finite numbers.')
            if round(value, 4) != value:
                raise ValueError('Expected four-decimal tweet-count sensitivity estimates.')
        if not 0 <= row['p_label'] <= 1:
            raise ValueError('Invalid tweet-count sensitivity p-value.')
    return pd.DataFrame(rows, columns=TWEET_COUNT_COLUMNS)


def save_tweet_count_results(table):
    record = {
        'schema_version': 1,
        'analysis': 'tweet_count_sensitivity',
        'source_notebook': '00_Data_and_Linguistic_Analysis.ipynb',
        'source_function': 'compute_tweet_count_sensitivity',
        'input_file': str(BASE / '16_linguistic_consistency/user_linguistic_consistency.csv'),
        'formula': 'y_rank ~ label + log_n',
        'round_decimals': 4,
        'rows': table.to_dict(orient='records'),
    }
    validate_tweet_count_record(record)
    payload = json.dumps(record, indent=2, allow_nan=False) + '\n'
    TWEET_COUNT_RESULTS.parent.mkdir(parents=True, exist_ok=True)
    with TWEET_COUNT_RESULTS.open('x', encoding='utf-8') as handle:
        handle.write(payload)


def ensure_tweet_count_input():
    global ENABLE_PREPROCESSING, PREPARATION_STAGE
    user_path = BASE / '16_linguistic_consistency/user_linguistic_consistency.csv'
    if user_path.is_file():
        return user_path
    tweet_path = BASE / '16_linguistic_consistency/tweet_linguistic_features.pkl'
    if not tweet_path.is_file():
        raise FileNotFoundError(
            f'Tweet-count input is unavailable: {user_path}\n'
            f'Building it requires: {tweet_path}\n'
            'Restore the tweet-feature artifact, or explicitly run features followed by user_features. '
            'Tweet-feature extraction is not started automatically.'
        )
    previous = ENABLE_PREPROCESSING, PREPARATION_STAGE
    try:
        ENABLE_PREPROCESSING, PREPARATION_STAGE = True, 'user_features'
        summarize_user_features()
    finally:
        ENABLE_PREPROCESSING, PREPARATION_STAGE = previous
    require_files([user_path])
    return user_path


def compute_tweet_count_sensitivity():
    require_source_stage('tweet_count', 'analysis', outputs=[str(TWEET_COUNT_RESULTS.relative_to(BASE))], inputs=[])
    ensure_tweet_count_input()
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    FEATURES = ['tweet_length', 'vocab_richness', 'sentiment_polarity', 'sentiment_subjectivity', 'pronoun_i_rate', 'pronoun_we_rate', 'exclamation_rate', 'question_rate', 'readability']
    LING_DIR = f'{SAVE_DIR}/16_linguistic_consistency'
    user_cons = pd.read_csv(BASE / '16_linguistic_consistency/user_linguistic_consistency.csv')
    from scipy.stats import spearmanr
    cons_cols = [c for c in user_cons.columns if c.endswith(('_sd', '_iqr'))]
    for col in cons_cols:
        rho, p = spearmanr(user_cons['n_tweets'], user_cons[col], nan_policy='omit')
    import statsmodels.formula.api as smf
    from scipy.stats import rankdata
    cons = user_cons.copy()
    cons['log_n'] = np.log(cons['n_tweets'])
    targets = ['tweet_length_sd', 'tweet_length_iqr', 'vocab_richness_sd', 'vocab_richness_iqr', 'pronoun_i_rate_iqr', 'sentiment_subjectivity_iqr']
    rows = []
    for col in targets:
        d = cons[[col, 'label', 'log_n']].dropna().copy()
        d['y_rank'] = rankdata(d[col]) / len(d)
        m = smf.ols('y_rank ~ label + log_n', data=d).fit()
        rows.append({'feature': col, 'coef_label': round(m.params['label'], 4), 'p_label': round(m.pvalues['label'], 4), 'coef_log_n': round(m.params['log_n'], 4)})
    pd.DataFrame(rows)
    result = pd.DataFrame(rows)
    save_tweet_count_results(result)
    return result


In [23]:
def compute_equal_count_analysis():
    require_source_stage('equal_count', 'analysis', outputs=['17_conference_revision/linguistic_robustness/dispersion_*.csv', '17_conference_revision/linguistic_robustness/equal_count_settings.csv'], inputs=['17_conference_revision/linguistic_robustness/tweet_linguistic_features_corrected.pkl'])
    import numpy as np
    import pandas as pd
    from pathlib import Path
    from scipy.stats import mannwhitneyu
    from statsmodels.stats.multitest import multipletests
    from tqdm.auto import tqdm
    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    OUT = BASE / '17_conference_revision/linguistic_robustness'
    OUT.mkdir(parents=True, exist_ok=True)
    FEATURES = ['tweet_length', 'vocab_richness', 'sentiment_polarity', 'sentiment_subjectivity', 'pronoun_i_rate', 'pronoun_we_rate', 'exclamation_rate', 'question_rate', 'readability']
    N_TWEETS = 500
    N_REPEATS = 100
    MASTER_SEED = 2026
    data = pd.read_pickle(OUT / 'tweet_linguistic_features_corrected.pkl')
    assert data[['username', 'label']].notna().all().all()
    assert data.groupby('username')['label'].nunique().eq(1).all()
    assert np.isfinite(data[FEATURES].to_numpy(dtype=float)).all()
    groups = []
    for username, group in data.groupby('username', sort=True):
        groups.append((int(group['label'].iloc[0]), group[FEATURES].to_numpy(dtype=float)))
    eligible = [(label, values) for label, values in groups if len(values) >= N_TWEETS]
    assert len(eligible) == 229

    def summarize_users(user_groups, rng=None):
        labels, dispersions = ([], [])
        for label, values in user_groups:
            if rng is not None:
                selected = rng.choice(len(values), size=N_TWEETS, replace=False)
                values = values[selected]
            sd = values.std(axis=0, ddof=1)
            quartiles = np.percentile(values, [25, 75], axis=0)
            iqr = quartiles[1] - quartiles[0]
            dispersions.append(np.concatenate([sd, iqr]))
            labels.append(label)
        return (np.asarray(labels), np.asarray(dispersions))

    def compare_groups(labels, dispersion):
        rows = []
        for j in range(dispersion.shape[1]):
            asd = dispersion[labels == 1, j]
            non_asd = dispersion[labels == 0, j]
            result = mannwhitneyu(asd, non_asd, alternative='two-sided', method='asymptotic')
            rows.append({'feature': FEATURES[j % len(FEATURES)], 'measure': 'SD' if j < len(FEATURES) else 'IQR', 'n_ASD': len(asd), 'n_nonASD': len(non_asd), 'median_ASD': np.median(asd), 'median_nonASD': np.median(non_asd), 'U': result.statistic, 'p_value': result.pvalue, 'rank_biserial_r': 2 * result.statistic / (len(asd) * len(non_asd)) - 1})
        table = pd.DataFrame(rows)
        table['p_FDR'] = multipletests(table['p_value'], method='fdr_bh')[1]
        table['significant_FDR_005'] = table['p_FDR'] < 0.05
        return table
    labels_all, dispersion_all = summarize_users(groups)
    baseline_all = compare_groups(labels_all, dispersion_all)
    baseline_all.to_csv(OUT / 'dispersion_all_244_users.csv', index=False)
    labels_eligible, dispersion_eligible = summarize_users(eligible)
    baseline_eligible = compare_groups(labels_eligible, dispersion_eligible)
    baseline_eligible.to_csv(OUT / 'dispersion_eligible_229_users_all_tweets.csv', index=False)
    repeat_tables = []
    repeat_seeds = np.random.SeedSequence(MASTER_SEED).generate_state(N_REPEATS)
    for repeat, seed in enumerate(tqdm(repeat_seeds, desc='Analysis'), start=1):
        rng = np.random.default_rng(int(seed))
        labels, dispersion = summarize_users(eligible, rng=rng)
        result = compare_groups(labels, dispersion)
        result.insert(0, 'repeat', repeat)
        result.insert(1, 'seed', int(seed))
        repeat_tables.append(result)
    repeated = pd.concat(repeat_tables, ignore_index=True)
    repeated.to_csv(OUT / 'dispersion_500tweets_100repeats.csv', index=False)
    summary = repeated.groupby(['feature', 'measure'], sort=False).agg(median_ASD=('median_ASD', 'median'), median_nonASD=('median_nonASD', 'median'), r_median=('rank_biserial_r', 'median'), r_p025=('rank_biserial_r', lambda x: x.quantile(0.025)), r_p975=('rank_biserial_r', lambda x: x.quantile(0.975)), fraction_FDR_significant=('significant_FDR_005', 'mean')).reset_index()
    summary = summary.merge(baseline_eligible[['feature', 'measure', 'rank_biserial_r', 'p_FDR']].rename(columns={'rank_biserial_r': 'r_eligible_all_tweets', 'p_FDR': 'p_FDR_eligible_all_tweets'}), on=['feature', 'measure'], validate='one_to_one')
    summary.to_csv(OUT / 'dispersion_500tweets_summary.csv', index=False)
    pd.DataFrame([{'tweets_per_user': N_TWEETS, 'n_repeats': N_REPEATS, 'master_seed': MASTER_SEED, 'n_users': len(eligible), 'sampling_with_replacement': False, 'FDR_family_size_per_repeat': 18, 'SD_ddof': 1}]).to_csv(OUT / 'equal_count_settings.csv', index=False)
    return summary


In [24]:
def compute_mattr_analysis():
    require_source_stage('mattr', 'analysis', outputs=['17_conference_revision/linguistic_robustness/MATTR_*'], inputs=['02_preprocessing/cleaned_data.pkl'])
    import json
    from collections import Counter
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from scipy.stats import ttest_ind
    from statsmodels.stats.multitest import multipletests
    from tqdm.auto import tqdm
    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    OUT = BASE / '17_conference_revision/linguistic_robustness'
    OUT.mkdir(parents=True, exist_ok=True)
    N_TWEETS = 500
    N_REPEATS = 20
    WINDOWS = [25, 50, 100]
    MASTER_SEED = 2027
    raw = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')
    assert raw[['username', 'label', 'cleaned_tweet']].notna().all().all()
    assert raw.groupby('username')['label'].nunique().eq(1).all()
    user_corpora = []
    for username, group in raw.groupby('username', sort=True):
        if len(group) >= N_TWEETS:
            tokens_per_tweet = [text.split() for text in group['cleaned_tweet'].astype(str)]
            user_corpora.append((int(group['label'].iloc[0]), tokens_per_tweet))
    assert len(user_corpora) == 229
    assert sum((label == 0 for label, _ in user_corpora)) == 24
    assert sum((label == 1 for label, _ in user_corpora)) == 205

    def mattr(tokens, window):
        if len(tokens) < window:
            raise ValueError('Required data or compatible artifacts are unavailable.')
        counts = Counter(tokens[:window])
        distinct = len(counts)
        total_distinct = distinct
        for i in range(window, len(tokens)):
            outgoing = tokens[i - window]
            counts[outgoing] -= 1
            if counts[outgoing] == 0:
                del counts[outgoing]
                distinct -= 1
            incoming = tokens[i]
            if incoming not in counts:
                distinct += 1
            counts[incoming] += 1
            total_distinct += distinct
        n_windows = len(tokens) - window + 1
        return total_distinct / (n_windows * window)
    assert np.isclose(mattr(['a', 'b', 'a', 'c'], 2), 1.0)
    assert np.isclose(mattr(['a', 'a', 'a'], 2), 0.5)
    repeat_seeds = np.random.SeedSequence(MASTER_SEED).generate_state(N_REPEATS)
    result_tables = []
    token_checks = []
    for repeat, seed in enumerate(tqdm(repeat_seeds, desc='Analysis'), start=1):
        rng = np.random.default_rng(int(seed))
        scores = {window: {0: [], 1: []} for window in WINDOWS}
        token_counts = []
        for label, tweets in user_corpora:
            selected = rng.choice(len(tweets), size=N_TWEETS, replace=False)
            tokens = [token for idx in selected for token in tweets[idx]]
            assert len(tokens) >= max(WINDOWS)
            token_counts.append(len(tokens))
            for window in WINDOWS:
                scores[window][label].append(mattr(tokens, window))
        rows = []
        for window in WINDOWS:
            asd = np.asarray(scores[window][1], dtype=float)
            non_asd = np.asarray(scores[window][0], dtype=float)
            test = ttest_ind(asd, non_asd, equal_var=False)
            df_effect = len(asd) + len(non_asd) - 2
            pooled_sd = np.sqrt(((len(asd) - 1) * asd.var(ddof=1) + (len(non_asd) - 1) * non_asd.var(ddof=1)) / df_effect)
            correction = 1 - 3 / (4 * df_effect - 1)
            g = correction * (asd.mean() - non_asd.mean()) / pooled_sd
            rows.append({'repeat': repeat, 'seed': int(seed), 'window': window, 'n_ASD': len(asd), 'n_nonASD': len(non_asd), 'ASD_mean': asd.mean(), 'non_ASD_mean': non_asd.mean(), 'difference': asd.mean() - non_asd.mean(), 't_stat': test.statistic, 'p_value': test.pvalue, 'hedges_g': g})
        table = pd.DataFrame(rows)
        table['p_FDR'] = multipletests(table['p_value'], method='fdr_bh')[1]
        table['significant_FDR_005'] = table['p_FDR'] < 0.05
        result_tables.append(table)
        token_checks.append({'repeat': repeat, 'min_tokens_per_user': min(token_counts), 'median_tokens_per_user': np.median(token_counts), 'max_tokens_per_user': max(token_counts)})
        pd.concat(result_tables, ignore_index=True).to_csv(OUT / 'MATTR_equal500_repeated_statistics.csv', index=False)
    results = pd.concat(result_tables, ignore_index=True)
    summary = results.groupby('window').agg(ASD_mean_median=('ASD_mean', 'median'), non_ASD_mean_median=('non_ASD_mean', 'median'), difference_median=('difference', 'median'), g_median=('hedges_g', 'median'), g_p025=('hedges_g', lambda x: x.quantile(0.025)), g_p975=('hedges_g', lambda x: x.quantile(0.975)), fraction_ASD_lower=('difference', lambda x: (x < 0).mean()), fraction_FDR_significant=('significant_FDR_005', 'mean')).reset_index()
    summary.to_csv(OUT / 'MATTR_equal500_summary.csv', index=False)
    pd.DataFrame(token_checks).to_csv(OUT / 'MATTR_token_counts_summary.csv', index=False)
    settings = {'tweets_per_user': N_TWEETS, 'n_repeats': N_REPEATS, 'windows': WINDOWS, 'primary_window': 50, 'master_seed': MASTER_SEED, 'n_ASD': 205, 'n_nonASD': 24, 'text_column': 'cleaned_tweet', 'tokenization': 'whitespace split', 'tweet_order': 'random within each repetition', 'windows_cross_tweet_boundaries': True, 'test': 'two-sided Welch t-test', 'FDR_family_size_per_repeat': 3}
    with open(OUT / 'MATTR_settings.json', 'w') as f:
        json.dump(settings, f, indent=2)
    return summary


In [25]:
def compute_topic_analysis():
    require_source_stage('topic_adjustment', 'analysis', outputs=['17_conference_revision/linguistic_robustness/topic_check/NMF10*', '17_conference_revision/linguistic_robustness/topic_check/topic_adjusted_dispersion_regression.csv', '17_conference_revision/linguistic_robustness/topic_check/PRIVATE_topic_covariates_and_dispersion.csv', '17_conference_revision/linguistic_robustness/topic_check/topic_covariate_group_summary.csv'], inputs=['02_preprocessing/cleaned_data.pkl', '17_conference_revision/linguistic_robustness/tweet_linguistic_features_corrected.pkl'])
    import json
    import warnings
    from pathlib import Path
    import joblib
    import numpy as np
    import pandas as pd
    import sklearn
    from sklearn.decomposition import NMF
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.exceptions import ConvergenceWarning
    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    OUT = BASE / '17_conference_revision/linguistic_robustness/topic_check'
    OUT.mkdir(parents=True, exist_ok=True)
    TOPIC_SEED = 2028
    topic_data = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl').reset_index(drop=True)
    assert topic_data[['username', 'label', 'cleaned_tweet']].notna().all().all()
    assert topic_data.groupby('username')['label'].nunique().eq(1).all()
    rng = np.random.default_rng(TOPIC_SEED)
    selected_positions = []
    for username, group in topic_data.groupby('username', sort=True):
        if len(group) >= 500:
            selected_positions.extend(rng.choice(group.index.to_numpy(), 500, replace=False))
    selected_positions = np.asarray(selected_positions, dtype=np.int64)
    assert len(selected_positions) == 229 * 500
    texts = topic_data.loc[selected_positions, 'cleaned_tweet'].astype(str)
    import json
    import warnings
    import nltk
    import numpy as np
    import pandas as pd
    import joblib
    from nltk import pos_tag_sents
    from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
    from sklearn.decomposition import NMF
    from sklearn.exceptions import ConvergenceWarning
    from tqdm.auto import tqdm
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    texts = topic_data.loc[selected_positions, 'cleaned_tweet'].astype(str).tolist()
    generic_words = {'thing', 'things', 'something', 'anything', 'everything', 'nothing', 'people', 'person', 'someone', 'anyone', 'way', 'ways', 'time', 'times', 'day', 'days', 'year', 'years', 'week', 'weeks', 'today', 'tomorrow', 'morning', 'night', 'lot', 'lots', 'thank', 'thanks', 'lol', 'omg', 'yeah', 'yes', 'xx', 've', 'll', 'don', 'didn', 'im'}
    excluded = set(ENGLISH_STOP_WORDS) | generic_words
    noun_documents = []
    BATCH_SIZE = 500
    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc='Analysis'):
        tokenized = [text.split() for text in texts[start:start + BATCH_SIZE]]
        tagged_batch = pos_tag_sents(tokenized, lang='eng')
        for tagged in tagged_batch:
            nouns = [word for word, tag in tagged if tag in {'NN', 'NNS'} and word.isalpha() and (len(word) >= 3) and (word not in excluded)]
            noun_documents.append(' '.join(nouns))
    noun_vectorizer = TfidfVectorizer(max_features=10000, min_df=10, max_df=0.8, ngram_range=(1, 1), sublinear_tf=True)
    X_nouns = noun_vectorizer.fit_transform(noun_documents)
    assert X_nouns.shape[1] >= 10
    noun_model = NMF(n_components=10, init='nndsvda', solver='cd', beta_loss='frobenius', max_iter=400, tol=0.0001, random_state=2028)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        noun_model.fit(X_nouns)
    not_converged = any((issubclass(w.category, ConvergenceWarning) for w in caught))
    terms = noun_vectorizer.get_feature_names_out()
    rows = []
    for topic_id, weights in enumerate(noun_model.components_):
        indices = np.argsort(weights)[-15:][::-1]
        rows.append({'topic': topic_id + 1, 'top_terms': ', '.join(terms[indices])})
    noun_topics = pd.DataFrame(rows)
    noun_topics.to_csv(OUT / 'NMF10_nouns_top_terms.csv', index=False)
    joblib.dump({'vectorizer': noun_vectorizer, 'model': noun_model}, OUT / 'NMF10_nouns_model.joblib')
    zero_fraction = float(np.mean(np.asarray(X_nouns.getnnz(axis=1)) == 0))
    with open(OUT / 'NMF10_nouns_settings.json', 'w') as f:
        json.dump({'n_topics': 10, 'seed': 2028, 'nltk_version': nltk.__version__, 'retained_POS': ['NN', 'NNS'], 'generic_words_removed': sorted(generic_words), 'min_word_length': 3, 'vectorizer': {'max_features': 10000, 'min_df': 10, 'max_df': 0.8, 'ngram_range': [1, 1], 'sublinear_tf': True}, 'nmf_max_iter': 400, 'nmf_tol': 0.0001, 'nmf_iterations': int(noun_model.n_iter_), 'convergence_warning': not_converged, 'zero_vector_fraction': zero_fraction}, f, indent=2)
    import numpy as np
    import pandas as pd
    import statsmodels.api as sm
    from scipy.stats import rankdata
    from statsmodels.stats.multitest import multipletests
    assert len(noun_documents) == len(selected_positions)
    assert len(np.unique(selected_positions)) == len(selected_positions)
    features = pd.read_pickle(BASE / '17_conference_revision/linguistic_robustness/tweet_linguistic_features_corrected.pkl').reset_index(drop=True)
    assert len(features) == len(topic_data)
    for col in ['username', 'label']:
        assert np.array_equal(features[col].to_numpy(), topic_data[col].to_numpy()), 'Artifact validation failed.'
    FEATURES = ['tweet_length', 'vocab_richness', 'sentiment_polarity', 'sentiment_subjectivity', 'pronoun_i_rate', 'pronoun_we_rate', 'exclamation_rate', 'question_rate', 'readability']
    sample = features.iloc[selected_positions].copy()
    assert sample.groupby('username').size().eq(500).all()
    assert sample['username'].nunique() == 229
    X_topic = noun_vectorizer.transform(noun_documents)
    weights = noun_model.transform(X_topic)
    valid = (np.asarray(X_topic.getnnz(axis=1)) > 0) & (weights.sum(axis=1) > 0)
    dominant_topic = np.full(len(sample), -1, dtype=int)
    dominant_topic[valid] = weights[valid].argmax(axis=1)
    sample['topic_id'] = dominant_topic
    sample['topic_valid'] = valid
    N_TOPICS = noun_model.n_components
    user_rows = []
    for username, group in sample.groupby('username', sort=True):
        valid_topics = group.loc[group['topic_valid'], 'topic_id'].to_numpy(dtype=int)
        assert len(valid_topics) > 0, 'Artifact validation failed.'
        counts = np.bincount(valid_topics, minlength=N_TOPICS)
        proportions = counts[counts > 0] / counts.sum()
        entropy = -np.sum(proportions * np.log(proportions)) / np.log(N_TOPICS)
        row = {'username': username, 'label': int(group['label'].iloc[0]), 'topic_entropy': entropy, 'topic_coverage': group['topic_valid'].mean()}
        for feature in FEATURES:
            values = group[feature].to_numpy(dtype=float)
            assert np.isfinite(values).all()
            row[f'{feature}__SD'] = values.std(ddof=1)
            row[f'{feature}__IQR'] = np.percentile(values, 75) - np.percentile(values, 25)
        user_rows.append(row)
    user_table = pd.DataFrame(user_rows)

    def standardized_ranks(values):
        ranked = rankdata(np.asarray(values), method='average')
        scale = ranked.std(ddof=1)
        assert scale > 0, 'Artifact validation failed.'
        return (ranked - ranked.mean()) / scale
    predictors = pd.DataFrame({'ASD_label': user_table['label'].astype(float), 'topic_entropy_rank': standardized_ranks(user_table['topic_entropy']), 'topic_coverage_rank': standardized_ranks(user_table['topic_coverage'])})
    X_unadjusted = sm.add_constant(predictors[['ASD_label']])
    X_adjusted = sm.add_constant(predictors)
    assert np.linalg.matrix_rank(X_adjusted.to_numpy()) == X_adjusted.shape[1]
    results = []
    for feature in FEATURES:
        for measure in ['SD', 'IQR']:
            outcome = standardized_ranks(user_table[f'{feature}__{measure}'])
            unadjusted = sm.OLS(outcome, X_unadjusted).fit(cov_type='HC3')
            adjusted = sm.OLS(outcome, X_adjusted).fit(cov_type='HC3')
            results.append({'feature': feature, 'measure': measure, 'n_users': len(user_table), 'beta_label_unadjusted': unadjusted.params['ASD_label'], 'p_unadjusted': unadjusted.pvalues['ASD_label'], 'beta_label_topic_adjusted': adjusted.params['ASD_label'], 'p_topic_adjusted': adjusted.pvalues['ASD_label']})
    regression_results = pd.DataFrame(results)
    for p_col, q_col in [('p_unadjusted', 'p_FDR_unadjusted'), ('p_topic_adjusted', 'p_FDR_topic_adjusted')]:
        regression_results[q_col] = multipletests(regression_results[p_col], method='fdr_bh')[1]
    regression_results.to_csv(OUT / 'topic_adjusted_dispersion_regression.csv', index=False)
    user_table.to_csv(OUT / 'PRIVATE_topic_covariates_and_dispersion.csv', index=False)
    coverage_summary = user_table.groupby('label')[['topic_entropy', 'topic_coverage']].agg(['min', 'median', 'max'])
    coverage_summary.to_csv(OUT / 'topic_covariate_group_summary.csv')
    return regression_results


In [26]:
stage_functions = {
    'tweet_count': compute_tweet_count_sensitivity,
    'equal_count': compute_equal_count_analysis,
    'mattr': compute_mattr_analysis,
    'topic_adjustment': compute_topic_analysis
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        if selected == 'tweet_count' and TWEET_COUNT_RESULTS.is_file():
            stage_result = validate_tweet_count_record(read_json(TWEET_COUNT_RESULTS))
        else:
            stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)
        if selected == 'tweet_count':
            print('Tweet-count results:', TWEET_COUNT_RESULTS)


,feature,coef_label,p_label,coef_log_n
0,tweet_length_sd,0.1635,0.0033,0.0681
1,tweet_length_iqr,0.1373,0.0144,0.0625
2,vocab_richness_sd,0.1359,0.0152,0.0663
3,vocab_richness_iqr,0.1105,0.0494,0.0650
4,pronoun_i_rate_iqr,0.1438,0.0096,0.0666
5,sentiment_subjectivity_iqr,-0.1618,0.0049,0.0167


Tweet-count results: /content/drive_research/MyDrive/thesis/saved_results/17_conference_revision/linguistic_robustness/tweet_count_sensitivity.json


### Linguistic Sensitivity Checks

The tweet_count stage computes and saves six regression rows to Drive. This section loads the generated results alongside equal-count, MATTR, and topic analyses.


In [27]:
def tweet_count_results():
    if not TWEET_COUNT_RESULTS.is_file():
        raise FileNotFoundError(
            f'{TWEET_COUNT_RESULTS}\n'
            "Run the tweet_count stage with ENABLE_ANALYSIS=True and ANALYSIS_STAGE='tweet_count'."
        )
    return validate_tweet_count_record(read_json(TWEET_COUNT_RESULTS))

def equal_count_results():
    settings = read_report_csv(LINGUISTIC / 'equal_count_settings.csv')
    display(settings)
    table = read_report_csv(LINGUISTIC / 'dispersion_500tweets_summary.csv')
    baseline = read_report_csv(LINGUISTIC / 'dispersion_all_244_users.csv')
    selected = baseline.loc[baseline['significant_FDR_005'], ['feature', 'measure']]
    table = selected.merge(table, on=['feature', 'measure'], validate='one_to_one')
    table['significant_repeats'] = (
        table['fraction_FDR_significant'] * int(settings.iloc[0]['n_repeats'])
    ).round().astype(int)
    return table.round(6)

def mattr_results():
    display(pd.json_normalize(read_json(LINGUISTIC / 'MATTR_settings.json')))
    table = read_report_csv(LINGUISTIC / 'MATTR_equal500_repeated_statistics.csv')
    assert not table.duplicated(['repeat', 'window']).any()
    return table.groupby('window').agg(
        repeats=('repeat', 'nunique'),
        ASD_mean_median=('ASD_mean', 'median'),
        non_ASD_mean_median=('non_ASD_mean', 'median'),
        FDR_significant_repeats=('significant_FDR_005', 'sum')
    ).reset_index().round(6)

def topic_results():
    table = read_report_csv(LINGUISTIC / 'topic_check/topic_adjusted_dispersion_regression.csv')
    baseline = read_report_csv(LINGUISTIC / 'dispersion_all_244_users.csv')
    selected = baseline.loc[baseline['significant_FDR_005'], ['feature', 'measure']]
    selected = selected.merge(table, on=['feature', 'measure'], validate='one_to_one')
    display(pd.DataFrame([{'originally_significant': len(selected),
                          'significant_after_topic_adjustment':
                          int(selected['p_FDR_topic_adjusted'].lt(0.05).sum())}]))
    return table.round(6)

def topic_coverage():
    return read_report_csv(LINGUISTIC / 'topic_check/topic_covariate_group_summary.csv',
               header=[0, 1], index_col=0).round(6)


In [28]:
show_result('Equal-Count Eligibility',lambda:equal_count_eligibility(data))
show_result('Tweet-Count Sensitivity',tweet_count_results)
show_result('Equal-Count Dispersion',equal_count_results)
show_result('MATTR',mattr_results)
show_result('Topic-Adjusted Dispersion',topic_results)
show_result('Topic Coverage',topic_coverage)


Equal-Count Eligibility


,tweets_per_user,eligible_users,ASD_users,non_ASD_users,excluded_users
0,500,229,205,24,15


Tweet-Count Sensitivity


,feature,coef_label,p_label,coef_log_n
0,tweet_length_sd,0.1635,0.0033,0.0681
1,tweet_length_iqr,0.1373,0.0144,0.0625
2,vocab_richness_sd,0.1359,0.0152,0.0663
3,vocab_richness_iqr,0.1105,0.0494,0.0650
4,pronoun_i_rate_iqr,0.1438,0.0096,0.0666
5,sentiment_subjectivity_iqr,-0.1618,0.0049,0.0167


Equal-Count Dispersion


,tweets_per_user,n_repeats,master_seed,n_users,sampling_with_replacement,FDR_family_size_per_repeat,SD_ddof
0,500,100,2026,229,False,18,1


... (6 outputs left out)

,feature,measure,n_users,beta_label_unadjusted,p_unadjusted,beta_label_topic_adjusted,p_topic_adjusted,p_FDR_unadjusted,p_FDR_topic_adjusted
0,tweet_length,SD,229,0.631596,0.001994,0.635756,0.005259,0.008972,0.013522
1,tweet_length,IQR,229,0.551795,0.005195,0.565926,0.014734,0.015497,0.033151
2,vocab_richness,SD,229,0.655482,0.001757,0.658016,0.002995,0.008972,0.010784
3,vocab_richness,IQR,229,0.577110,0.001045,0.584493,0.003728,0.008972,0.011184
4,sentiment_polarity,SD,229,-0.571176,0.006027,-0.601708,0.000064,0.015497,0.000292
5,sentiment_polarity,IQR,229,-0.565979,0.001617,-0.619317,0.000064,0.008972,0.000292
6,sentiment_subjectivity,SD,229,-0.546587,0.011730,-0.563850,0.000065,0.026393,0.000292
7,sentiment_subjectivity,IQR,229,-0.627070,0.003138,-0.641530,0.000029,0.011298,0.000292
8,pronoun_i_rate,SD,229,0.155264,0.545793,0.119438,0.522336,0.818690,0.783504
9,pronoun_i_rate,IQR,229,0.487768,0.017468,0.440543,0.032246,0.034936,0.064491


Topic Coverage


topic_entropy                     topic_coverage              
                min    median       max            min median    max
label                                                               
0          0.287434  0.682352  0.799428          0.610  0.801  0.980
1          0.344319  0.685814  0.855146          0.426  0.810  0.994

topic_entropy                     topic_coverage              
                min    median       max            min median    max
label                                                               
0          0.287434  0.682352  0.799428          0.610  0.801  0.980
1          0.344319  0.685814  0.855146          0.426  0.810  0.994